# Smart Retail Intelligence System
## Notebook 1: Customer Segmentation and High-Value Prediction

**Dataset:** UCI Online Retail II (an online shop in the UK, Dec 2009 to Dec 2011)

| Part | What we do | Syllabus |
|---|---|---|
| 1 | Clean the data and build one row per customer | Module 1 |
| 2 | Group customers with K-Means | Module 4 |
| 3 | Predict high-value customers with 4 models and compare them | Modules 3 and 5 |

**How to run:** put `online_retail_II.xlsx` in your Drive folder `aiml_minproject`, then choose **Runtime > Run all**.

In [ ]:
import os
import pandas as pd

DATA_DIR = "./data"
FILE_PATH = os.path.join(DATA_DIR, "online_retail.csv")

df = pd.read_csv(FILE_PATH)

print(df.head())
print(df.shape)

: 

In [ ]:
# Step 2: import the libraries we need
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Part 1: Prepare the data
We remove cancelled orders, returns, wrong prices and non-product codes (like postage).

**Important idea: split by time.** Customer behaviour **before 9 Sep 2011** is used to build the inputs. What customers spend **after** that date is what we try to predict. This way the model never sees the future.

In [ ]:
CUTOFF = pd.Timestamp("2011-09-09")

# Load the data (the first time takes a few minutes, later runs use the saved copy)
if os.path.exists("retail_raw.pkl"):
    raw = pd.read_pickle("retail_raw.pkl")
else:
    sheets = pd.read_excel("online_retail_II.xlsx", sheet_name=None)
    raw = pd.concat(sheets.values(), ignore_index=True)
    raw.to_pickle("retail_raw.pkl")

df = raw.copy()
df.columns = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "CustomerID", "Country"]
df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

rows_before = len(df)
df = df[~df["Invoice"].str.startswith("C")]    # remove cancelled invoices (they start with C)
df = df[df["Quantity"] > 0]                    # remove returns
df = df[df["Price"] > 0]                       # remove wrong prices
df = df[df["StockCode"].str.match(r"^\d{5}")]  # keep normal product codes (5 digits)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["Price"]
print("Rows before cleaning:", rows_before, "| after cleaning:", len(df))

In [ ]:
# Keep only rows that have a customer, then split into "before" and "after" the cutoff date
df_cust = df.dropna(subset=["CustomerID"]).copy()
df_cust["CustomerID"] = df_cust["CustomerID"].astype(int)

feature_df = df_cust[df_cust["InvoiceDate"] < CUTOFF]    # used to build inputs
target_df = df_cust[df_cust["InvoiceDate"] >= CUTOFF]    # used to build what we predict
print("Rows before cutoff:", len(feature_df), "| rows after cutoff:", len(target_df))

### One row per customer
- **Recency**: days since the last purchase (small = bought recently)
- **Frequency**: number of orders
- **Monetary**: total money spent
- **NumProducts**: number of different products bought
- **AvgOrderValue**: money spent per order
- **FutureSpend**: money spent after the cutoff date (0 if the customer did not come back)
- **HighValue**: 1 if the customer is in the top 25% of FutureSpend, otherwise 0

In [ ]:
customers = feature_df.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("Invoice", "nunique"),
    Monetary=("Revenue", "sum"),
    NumProducts=("StockCode", "nunique"))

customers["Recency"] = (CUTOFF - customers["LastPurchase"]).dt.days
customers["AvgOrderValue"] = customers["Monetary"] / customers["Frequency"]

# What customers spent AFTER the cutoff (customers who did not buy again get 0)
future = target_df.groupby("CustomerID")["Revenue"].sum()
customers["FutureSpend"] = future
customers["FutureSpend"] = customers["FutureSpend"].fillna(0)

# High value = top 25% of future spenders
threshold = customers["FutureSpend"].quantile(0.75)
customers["HighValue"] = (customers["FutureSpend"] > threshold).astype(int)

# Money columns are very skewed (a few customers spend a lot), so we also make "log" versions
for col in ["Frequency", "Monetary", "NumProducts", "AvgOrderValue"]:
    customers[col + "_log"] = np.log1p(customers[col])

print("Number of customers:", len(customers))
print("High-value threshold (spend):", round(threshold, 2))
print("Share of customers who bought again:", round((customers["FutureSpend"] > 0).mean(), 3))
print(customers["HighValue"].value_counts())
customers.head()

## Part 2: Group customers with K-Means (Module 4)
K-Means puts similar customers in the same group. We first **scale** the columns so they are on the same range. Then we try different numbers of groups (k):
- **Elbow method:** look for the point where the curve stops dropping quickly.
- **Silhouette score:** higher means better separated groups.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_cols = ["Recency", "Frequency_log", "Monetary_log", "NumProducts_log", "AvgOrderValue_log"]
X = StandardScaler().fit_transform(customers[cluster_cols])

k_values = [2, 3, 4, 5, 6, 7, 8, 9, 10]
inertia_list = []
silhouette_list = []
for k in k_values:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(X)
    inertia_list.append(model.inertia_)
    silhouette_list.append(silhouette_score(X, model.labels_))

plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
plt.plot(k_values, inertia_list, marker="o")
plt.title("Elbow method")
plt.xlabel("k (number of groups)")
plt.subplot(1, 2, 2)
plt.plot(k_values, silhouette_list, marker="o")
plt.title("Silhouette score")
plt.xlabel("k (number of groups)")
plt.show()

**We choose k = 4.** The elbow curve bends slowly around 3 to 4. The silhouette score is highest at k = 2, but only two groups (active and inactive) is too simple to be useful, so we use 4 groups.

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
customers["Segment"] = kmeans.fit_predict(X)

# Describe each group by its averages
customers["BoughtAgain"] = (customers["FutureSpend"] > 0).astype(int)
profile = customers.groupby("Segment")[["Recency", "Frequency", "Monetary", "AvgOrderValue", "BoughtAgain"]].mean()
profile["Customers"] = customers["Segment"].value_counts()

# Give names by total spending: lowest spending group first
order = profile.sort_values("Monetary").index
labels = ["Low spenders", "Medium spenders", "High spenders", "Top spenders"]
names = {}
for i in range(4):
    names[order[i]] = labels[i]
customers["SegmentName"] = customers["Segment"].map(names)
profile["Name"] = profile.index.map(names)
profile.round(2)

In [ ]:
# PCA squeezes the 5 columns into 2 so we can draw the groups on a flat plot
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
points = pca.fit_transform(X)
print("Information kept by the 2 PCA components:", round(pca.explained_variance_ratio_.sum() * 100, 1), "%")

plt.figure(figsize=(7, 5))
for name in labels:
    mask = (customers["SegmentName"] == name).values
    plt.scatter(points[mask, 0], points[mask, 1], s=8, label=name)
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.title("Customer groups")
plt.legend()
plt.show()

**Reading the table:** the BoughtAgain column goes up as spending goes up. We did not use FutureSpend to make the groups, so this shows the groups are meaningful.

## Part 3: Predict high-value customers (Modules 3 and 5)
Question: *will this customer be a high-value customer in the next 3 months?*

We split the customers into **80% training data** (to learn from) and **20% test data** (to check the model on customers it has not seen).

Only 25% of customers are high-value, so a model that always says "no" gets 75% accuracy. This is our **baseline**. Accuracy alone is not enough, so we also look at **precision, recall, F1 and ROC-AUC**.

In [ ]:
from sklearn.model_selection import train_test_split

inputs = ["Recency", "Frequency_log", "Monetary_log", "NumProducts_log", "AvgOrderValue_log"]
X_all = customers[inputs]
y_all = customers["HighValue"]

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)

# Scale using the training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Training customers:", len(y_train), "| test customers:", len(y_test))

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# "balanced" gives more importance to the rare high-value customers
models = {
    "Baseline (always 0)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
}

results = []
for name in models:
    model = models[name]
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    test_prob = model.predict_proba(X_test)[:, 1]
    cv_f1 = cross_val_score(model, X_train, y_train, cv=5, scoring="f1").mean()   # 5-fold cross-validation

    results.append({
        "Model": name,
        "Train F1": f1_score(y_train, train_pred),
        "CV F1": cv_f1,
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Test Precision": precision_score(y_test, test_pred, zero_division=0),
        "Test Recall": recall_score(y_test, test_pred),
        "Test F1": f1_score(y_test, test_pred),
        "Test ROC-AUC": roc_auc_score(y_test, test_prob)})

results_table = pd.DataFrame(results)
results_table.round(3)

**How to read the table**
- Compare every model with the **baseline**: look at F1 and ROC-AUC, not only accuracy.
- **Overfitting:** if *Train F1* is much higher than *CV F1* or *Test F1*, the model memorised the training data. A Decision Tree and a Random Forest often do this.
- **Precision** = of the customers we flag as high-value, how many really are. **Recall** = of all real high-value customers, how many we find.

In [ ]:
# Confusion matrix: rows = real answer, columns = model's answer
from sklearn.metrics import ConfusionMatrixDisplay

plt.figure(figsize=(15, 3.5))
position = 1
for name in ["Logistic Regression", "KNN", "Decision Tree", "Random Forest"]:
    plt.subplot(1, 4, position)
    ConfusionMatrixDisplay.from_predictions(y_test, models[name].predict(X_test), ax=plt.gca(), colorbar=False)
    plt.title(name)
    position = position + 1
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve: the closer the line is to the top-left corner, the better the model
from sklearn.metrics import roc_curve

plt.figure(figsize=(6, 5))
for name in ["Logistic Regression", "KNN", "Decision Tree", "Random Forest"]:
    probs = models[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=name + " (AUC " + str(round(roc_auc_score(y_test, probs), 2)) + ")")
plt.plot([0, 1], [0, 1], "k--", label="Random guessing")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.legend()
plt.show()

### Hyperparameter tuning (Random Forest)
**GridSearchCV** tries every combination of the settings below, checks each with cross-validation, and keeps the best one. Then we compare the default model with the tuned model.

In [ ]:
from sklearn.model_selection import GridSearchCV

settings = {
    "n_estimators": [100, 300],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 5]}

grid = GridSearchCV(RandomForestClassifier(class_weight="balanced", random_state=42), settings, cv=5, scoring="f1")
grid.fit(X_train, y_train)
print("Best settings:", grid.best_params_)

default_rf = models["Random Forest"]
tuned_rf = grid.best_estimator_

compare = []
for name, model in [("Default Random Forest", default_rf), ("Tuned Random Forest", tuned_rf)]:
    pred = model.predict(X_test)
    compare.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])})
pd.DataFrame(compare).round(3)

In [ ]:
# Which inputs matter most for the tuned model?
importance = pd.Series(tuned_rf.feature_importances_, index=inputs).sort_values()
importance.plot(kind="barh", title="Feature importance (tuned Random Forest)")
plt.show()

## Save files for Notebook 2
Notebook 2 needs a table showing which products were bought together on each invoice (the "basket" table) and the product names. We keep the 150 most-bought products so the programs run quickly.

In [ ]:
top_products = df["StockCode"].value_counts().head(150).index
basket_data = df[df["StockCode"].isin(top_products)]

# Rows = invoices, columns = products, value True if the product is on the invoice
basket = basket_data.groupby(["Invoice", "StockCode"])["Quantity"].sum().unstack(fill_value=0) > 0
basket = basket[basket.sum(axis=1) >= 2]       # keep invoices with at least 2 products

names_table = df.groupby("StockCode")["Description"].agg(lambda s: s.mode().iloc[0])
product_names = names_table.str.strip().str.title().reindex(top_products)

basket.to_pickle("basket.pkl")
product_names.to_pickle("product_names.pkl")
print("Basket table:", basket.shape, "| files saved: basket.pkl, product_names.pkl")

## Summary
- K-Means found **4 customer groups**, drawn on a PCA plot.
- Four models predicted high-value customers and were compared with a baseline using accuracy, precision, recall, F1, ROC-AUC, a confusion matrix and cross-validation.
- Tuning the Random Forest with GridSearchCV changed its results (compare the two rows above).

**Limitations:** one UK shop only; the prediction period is just 3 months and includes the holiday season; many customers never come back, so the target is noisy.